In [1]:
!pip install -q transformers==4.46.0 peft bitsandbytes accelerate jamotools jamo trl -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.7 MB/s eta 0:00:00


In [6]:
import os
import io
import gc
import base64
import torch
import torchaudio
import jamotools
from jamo import h2j, j2hcj
from IPython.display import display, Javascript
from google.colab import output, drive
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, AutoModelForSequenceClassification
)
from peft import PeftModel

# 1. Mount Storage
drive.mount('/content/drive')

# 2. Performance Tracking Hardware Verification
print(f"✅ GPU Active: {torch.cuda.is_available()}")
print(f"🖥️ Execution Device: {torch.cuda.get_device_name(0)}")

# 3. SET GLOBALS: Points directly to your finalized model assets in Drive
SAVE_DIR = "/content/drive/MyDrive/manual_datasets/dialogue_system2"
ASR_MODEL_PATH = "/content/drive/MyDrive/manual_datasets/clovacall_data/final_asr_v3_model"
ROUTER_PATH = os.path.join(SAVE_DIR, "intent_router")
RESTAURANT_EXPERT_PATH = os.path.join(SAVE_DIR, "restaurant_expert")
TRAVEL_EXPERT_PATH = os.path.join(SAVE_DIR, "travel_expert")
COHERENCE_EXPERT_PATH = os.path.join(SAVE_DIR, "coherence_expert")
SLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Define your base directory for audio
audio_dir = '/content/drive/MyDrive/audio_files/'

print("✅ Configuration paths mapped successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU Active: True
🖥️ Execution Device: Tesla T4
✅ Configuration paths mapped successfully!


In [3]:
import pandas as pd

def check_semantic_context_anomaly(student_transcription):
    """
    Checks if the student's vocalized transcription matches a known
    KoSReS semantic abnormality, indicating incoherent language production.
    """
    # Point directly to your uploaded dataset file path
    file_path = "/content/drive/MyDrive/manual_datasets/S1_JSLHR-23-00137song.xlsx"

    try:
        df_anomalous = pd.read_excel(file_path, sheet_name='anomalous')
        df_anomalous.columns = ['Sentence Code', 'Sentence']
        df_anomalous = df_anomalous[df_anomalous['Sentence Code'] != 'Sentence Code'].reset_index(drop=True)
        anomalous_corpus = df_anomalous['Sentence'].tolist()

        clean_hyp = student_transcription.replace(" ", "")
        for nonsense_sentence in anomalous_corpus:
            clean_nonsense = str(nonsense_sentence).replace(" ", "")
            if clean_nonsense in clean_hyp:
                return True # Out-of-bounds semantic anomaly detected!
    except Exception as e:
        print(f"⚠️ Metadata check fallback alert: {e}")

    return False

In [7]:
!ls "/content/drive/MyDrive/audio_files/"

'감사합니다, 정말 맛있어요.m4a'  '비빔밥 하나 주세요.m4a'
'물 좀 주세요.m4a'		 '여기 명소 추천해주세요.m4a'
'박물관이 어디에 있어요.m4a'


In [8]:
wav_dir = '/content/wav_files/'
os.makedirs(wav_dir, exist_ok=True)

# 3. Batch Convert
# This uses FFmpeg to ensure every file is 16kHz, Mono, WAV
files = [f for f in os.listdir(audio_dir) if f.endswith('.m4a')]
for filename in files:
    input_path = os.path.join(audio_dir, filename)
    output_path = os.path.join(wav_dir, filename.replace('.m4a', '.wav'))
    os.system(f"ffmpeg -i '{input_path}' -ar 16000 -ac 1 '{output_path}' -y")
    print(f"✅ Prepared: {filename} -> {os.path.basename(output_path)}")

✅ Prepared: 여기 명소 추천해주세요.m4a -> 여기 명소 추천해주세요.wav
✅ Prepared: 물 좀 주세요.m4a -> 물 좀 주세요.wav
✅ Prepared: 박물관이 어디에 있어요.m4a -> 박물관이 어디에 있어요.wav
✅ Prepared: 감사합니다, 정말 맛있어요.m4a -> 감사합니다, 정말 맛있어요.wav
✅ Prepared: 비빔밥 하나 주세요.m4a -> 비빔밥 하나 주세요.wav


In [9]:
def process_saved_experiment(wav_files_path, target_sentence):
    """Processes a single file and outputs the diagnostic logs."""
    # Run diagnostics
    transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(wav_files_path, target_sentence)

    # Run the Gating Node and Expert Generation
    response = generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence)

    # Print clean results for your thesis logs
    print("=" * 60)
    print(f"📝 Expected: {target_sentence}")
    print(f"📝 Heard:    {transcription}")
    print(f"📊 Accuracy: {(1.0 - per)*100:.1f}%")
    print(f"🤖 Feedback: {response}")
    print("=" * 60)

In [10]:
print("📦 Loading Acoustic Signal Processors...")
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor

VOCAB_PATH = "/content/drive/MyDrive/manual_datasets/clovacall_data/jamo_vocab.json"

tokenizer         = Wav2Vec2CTCTokenizer(VOCAB_PATH, unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(ASR_MODEL_PATH)
asr_processor     = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
asr_model         = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_PATH).to("cuda")
asr_model.eval()

print("🧭 Loading Gating Intent Router...")
router_tokenizer = AutoTokenizer.from_pretrained("klue/roberta-small")
router_model     = AutoModelForSequenceClassification.from_pretrained(ROUTER_PATH).to("cuda")
router_model.eval()

print("✅ All models loaded!")

📦 Loading Acoustic Signal Processors...
🧭 Loading Gating Intent Router...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/971 [00:00<?, ?B/s]

✅ All models loaded!


In [11]:
def compute_edit_distance(ref, hyp):
    N, M = len(ref), len(hyp)
    dp = [[0] * (M + 1) for _ in range(N + 1)]
    for i in range(N + 1): dp[i][0] = i
    for j in range(M + 1): dp[0][j] = j
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    return dp

def backtrack(dp, ref, hyp):
    i, j = len(ref), len(hyp)
    operations = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            operations.append({"position": i, "type": "substitution", "expected": ref[i-1], "predicted": hyp[j-1]})
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            operations.append({"position": i, "type": "deletion", "expected": ref[i-1], "predicted": "[missing]"})
            i -= 1;
        else:
            operations.append({"position": j, "type": "insertion", "expected": "[none]", "predicted": hyp[j-1]})
            j -= 1
    operations.reverse()
    return operations

def get_pronunciation_diagnostics(audio_file, target_hangul):
    speech, sr = torchaudio.load(audio_file)
    if sr != 16000:
        speech = torchaudio.transforms.Resample(sr, 16000)(speech)

    input_values = asr_processor(speech.squeeze().numpy(), sampling_rate=16000, return_tensors="pt").input_values.to("cuda")
    with torch.no_grad():
        logits = asr_model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_jamo_str = asr_processor.batch_decode(pred_ids)[0]
    transcription = jamotools.join_jamos(pred_jamo_str).strip()

    ref_clean, hyp_clean = target_hangul.replace(" ", ""), transcription.replace(" ", "")
    ref_jamo, hyp_jamo = list(jamotools.split_syllables(ref_clean)), list(jamotools.split_syllables(hyp_clean))

    dp = compute_edit_distance(ref_jamo, hyp_jamo)
    per = (dp[len(ref_jamo)][len(hyp_jamo)] / len(ref_jamo)) if ref_jamo else 0.0

    ops = backtrack(dp, ref_jamo, hyp_jamo)
    s = sum(1 for o in ops if o["type"] == "substitution")
    d = sum(1 for o in ops if o["type"] == "deletion")
    i = sum(1 for o in ops if o["type"] == "insertion")

    syl_dp = compute_edit_distance(list(ref_clean), list(hyp_clean))
    syl_errors = backtrack(syl_dp, list(ref_clean), list(hyp_clean))
    print(f"🔍 Syllable errors: {syl_errors}")

    return transcription, per, syl_errors, (s, d, i)

def route_intent(text):
    """Routes the user transcription to the correct domain expert adapter safely."""
    encoding = router_tokenizer(
        text,
        max_length=64,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids      = encoding['input_ids'].to("cuda")
    attention_mask = encoding['attention_mask'].to("cuda")

    with torch.no_grad():
        outputs = router_model(input_ids=input_ids, attention_mask=attention_mask)

    # 🔄 FIXED: Check if outputs is a dictionary or a tuple to prevent AttributeError
    if hasattr(outputs, "logits"):
        logits = outputs.logits
    else:
        logits = outputs[0]  # If it returns a raw tuple, logits sit at index 0

    probs = torch.softmax(logits, dim=-1)
    pred  = torch.argmax(probs, dim=-1).item()

    return "restaurant" if pred == 0 else "travel"

In [12]:
def load_expert(expert_path):
    """Dynamic model loader implementing strict garbage collection to safeguard VRAM boundaries."""
    global expert_model
    if 'expert_model' in globals():
        del expert_model
    gc.collect()
    torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        SLM_MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    return PeftModel.from_pretrained(base_model, expert_path)

def generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence):
    """
    Enhanced MoE Response Generator incorporating the KoSReS Context Filter
    to detect semantic abnormalities before standard intent routing.
    """
    # 🚨 STEP 1: Run the KoSReS Semantic Context Anomaly Check
    is_anomaly = check_semantic_context_anomaly(transcription)

    # 🚨 STEP 2: Determine the active adapter path and system role configuration
    if is_anomaly:
        print("🚨 [GATING NODE INTERVENTION]: Out-of-Bounds Semantic Anomaly Detected via KoSReS Check.")
        expert_path = COHERENCE_EXPERT_PATH
        role = "conversational coherence monitor"
    else:
        # Standard scenario intent routing
        intent = route_intent(transcription)
        print(f"🧭 Gating Node Target Match: [{intent.upper()} EXPERT]")
        expert_path = RESTAURANT_EXPERT_PATH if intent == "restaurant" else TRAVEL_EXPERT_PATH
        role = "restaurant host in Seoul" if intent == "restaurant" else "tour guide in Northern Seoul"

    # 🧼 STEP 3: Strict garbage collection routine to stay within T4 hardware VRAM boundaries
    global expert_model
    if 'expert_model' in globals():
        del expert_model
    gc.collect()
    torch.cuda.empty_cache()

    # 🧼 STEP 4: Initialize the 4-bit unadapted base model matching the correct float16 compute tensors
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16  # Eliminates NotImplementedError conflicts
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        SLM_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # Load the targeted LoRA adapter config flagged by the Gating Node step
    expert_model = PeftModel.from_pretrained(base_model, expert_path)
    slm_tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL_ID, trust_remote_code=True)

    s, d, i = error_counts

    # 📝 STEP 5: Context Prompt Construction Matrix
    feedback_sentence = "Your pronunciation was excellent and clear!"
    if syl_errors:
        sub_errors = [err for err in syl_errors if err.get("type") == "substitution"]
        if sub_errors:
            details = [f"'{err['predicted']}' instead of '{err['expected']}'" for err in sub_errors]
            feedback_sentence = "You accidentally said " + " and ".join(details) + "."

    # Separate logic flows for standard scenarios vs. semantic anomaly safety intervention
    if is_anomaly:
        system_instruction = (
            f"You are an expert AI Korean Pronunciation Coach roleplaying as a {role}. Speak ONLY in English.\n\n"
            f"SITUATION: A Korean language student just spoke a semantically nonsensical or anomalous statement to you.\n"
            f"WHAT YOU HEARD: '{transcription}'\n\n"
            f"YOUR TASK:\n"
            f"1. In 1 sentence, inform them that a semantic/contextual error was detected in their phrase logic.\n"
            f"2. In 1 sentence, politely guide them back to practicing standard scenario phrases (like asking for water or museum directions).\n"
            f"Do not invent corrections for the nonsense phrase. Focus on conversational remediation."
        )
    elif per == 0.0:
        system_instruction = (
            f"You are an expert AI Korean Pronunciation Coach roleplaying as a {role}. Speak ONLY in English.\n\n"
            f"SITUATION: A Korean language student just spoke to you perfectly.\n"
            f"WHAT THEY SAID: '{transcription}'\n\n"
            f"YOUR TASK:\n"
            f"1. In 1 sentence, praise their perfect pronunciation.\n"
            f"2. In 1 sentence, respond in character to their request.\n"
            f"IMPORTANT: Do NOT mention any errors. There were none. Do not invent corrections."
        )
    else:
        system_instruction = (
            f"You are an expert AI Korean Pronunciation Coach roleplaying as a {role}. Speak ONLY in English.\n\n"
            f"SITUATION: A Korean language student just spoke to you.\n"
            f"WHAT THEY WERE SUPPOSED TO SAY: '{target_sentence}'\n"
            f"WHAT YOU HEARD: '{transcription}'\n"
            f"PRONUNCIATION FEEDBACK: {feedback_sentence}\n\n"
            f"YOUR TASK:\n"
            f"1. In 1 sentence, gently tell the student which sound they got wrong using the PRONUNCIATION FEEDBACK above.\n"
            f"2. In 1 sentence, respond in character to their request.\n"
            f"Do not apologize. Do not pretend to be the student. You are the coach and {role.split()[0]}."
        )

    user_payload = f"Target: {target_sentence}\nTranscription: {transcription}\nGenerate the response now."

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user",   "content": user_payload}
    ]
    prompt = slm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = slm_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
      outputs = expert_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,          # Required for sampling
        temperature=0.1,         # Low temperature = high confidence
        top_p=0.9,               # Nucleus sampling
        top_k=20,                # Restrict sampling to top-k candidates
        pad_token_id=slm_tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return slm_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

In [16]:
import os

wav_dir = '/content/wav_files/'
# 1. Get current files
current_files = sorted([f for f in os.listdir(wav_dir) if f.endswith('.wav')])

# 2. Map the new simple names to your target sentences
# This order matches the alphabetical sort of your files
experiment_map = {
    "test_0.wav": "감사합니다, 정말 맛있어요.", # Originally started with '감'
    "test_1.wav": "물 좀 주세요.",            # Originally started with '물'
    "test_2.wav": "박물관이 어디에 있어요?",     # Originally started with '박'
    "test_3.wav": "비빔밥 하나 주세요",        # Originally started with '비'
    "test_4.wav": "여기 명소 추천해주세요."     # Originally started with '여'
}

# 3. Rename files and Run pipeline
for i, old_name in enumerate(current_files):
    new_name = f"test_{i}.wav"
    os.rename(os.path.join(wav_dir, old_name), os.path.join(wav_dir, new_name))
    print(f"✅ Renamed: {old_name} -> {new_name}")

print("\n🚀 Starting Pipeline with clean filenames...")

# 4. Now run the loop using the new names
for filename, target in experiment_map.items():
    file_path = os.path.join(wav_dir, filename)

    print(f"\n🎯 Processing Objective: {target}")
    print("-" * 50)

    # Run pipeline
    transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(file_path, target)
    response = generate_tutor_response(transcription, per, syl_errors, error_counts, target)

    # Output
    print("=" * 60)
    print(f"📝 Expected: {target}")
    print(f"📝 Heard:    {transcription}")
    print(f"📊 Accuracy: {(1.0 - per)*100:.1f}%")
    print(f"🤖 Feedback: {response}")
    print("=" * 60)

✅ Renamed: 감사합니다, 정말 맛있어요.wav -> test_0.wav
✅ Renamed: 물 좀 주세요.wav -> test_1.wav
✅ Renamed: 박물관이 어디에 있어요.wav -> test_2.wav
✅ Renamed: 비빔밥 하나 주세요.wav -> test_3.wav
✅ Renamed: 여기 명소 추천해주세요.wav -> test_4.wav

🚀 Starting Pipeline with clean filenames...

🎯 Processing Objective: 감사합니다, 정말 맛있어요.
--------------------------------------------------
🔍 Syllable errors: [{'position': 6, 'type': 'substitution', 'expected': ',', 'predicted': '.'}, {'position': 7, 'type': 'substitution', 'expected': '정', 'predicted': '종'}, {'position': 8, 'type': 'substitution', 'expected': '말', 'predicted': '마'}, {'position': 9, 'type': 'substitution', 'expected': '맛', 'predicted': '마'}, {'position': 10, 'type': 'substitution', 'expected': '있', 'predicted': '싰'}, {'position': 13, 'type': 'deletion', 'expected': '.', 'predicted': '[missing]'}]
🧭 Gating Node Target Match: [TRAVEL EXPERT]


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

📝 Expected: 박물관이 어디에 있어요?
📝 Heard:    박무율권이 어디에 있어요?
📊 Accuracy: 88.0%
🤖 Feedback: Your pronunciation is correct! The answer is "박물관이 어디에 있어요?"

🎯 Processing Objective: 비빔밥 하나 주세요
--------------------------------------------------
🔍 Syllable errors: [{'position': 3, 'type': 'substitution', 'expected': '밥', 'predicted': '벅'}, {'position': 4, 'type': 'substitution', 'expected': '하', 'predicted': '바'}]
🧭 Gating Node Target Match: [RESTAURANT EXPERT]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


📝 Expected: 비빔밥 하나 주세요
📝 Heard:    비빔벅 바나주세요
📊 Accuracy: 83.3%
🤖 Feedback: Your pronunciation is almost correct! The word should end with "밥" instead of "벅". Please try again.

🎯 Processing Objective: 여기 명소 추천해주세요.
--------------------------------------------------
🔍 Syllable errors: []
🧭 Gating Node Target Match: [TRAVEL EXPERT]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


📝 Expected: 여기 명소 추천해주세요.
📝 Heard:    여기명소 추천해주세요.
📊 Accuracy: 100.0%
🤖 Feedback: Thank you for your perfect pronunciation! I'm happy to recommend some places of interest. Please let me know what kind of place you're interested in.


In [14]:
!ls /content/wav_files/

'감사합니다, 정말 맛있어요.wav'  '비빔밥 하나 주세요.wav'
'물 좀 주세요.wav'		 '여기 명소 추천해주세요.wav'
'박물관이 어디에 있어요.wav'
